##### Imports

In [1]:
# %pip install -r requirements.txt

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple
from scipy.stats import norm, invgamma, multivariate_normal
import pickle
import os

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

warnings.filterwarnings('ignore')


from kama_msr import KAMA
from kama_msr import MarkovSwitchingModel
from kama_msr import KAMA_MSR

# Getting Price Data

#### -------------------------------------------------------------------------------------------------

In [3]:
# Prepare data
international_index_symbol_names = pd.read_csv('data/inputs/fmp_index_list.csv').set_index('symbol')['name']
international_index_symbol_names = international_index_symbol_names[~international_index_symbol_names.index.isin(['^GSPC', '^NDX'])].to_dict()
commodity_symbol_names = pd.read_csv('data/inputs/fmp_commodity_list.csv').set_index('symbol')['name'].to_dict()
etf_symbol_names = {
    # BOND ETFS
    'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    'IEF': 'iShares 7-10 Year Treasury Bond ETF',
    # International EQUITY ETFS
    'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    '^RUI': 'Russell 1000',
    '^RUA': 'Russell 3000',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    'VOO': 'Vanguard S&P 500 ETF',
    'RSP': 'Invesco S&P 500 Equal Weight ETF',
    'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'QQQM': 'Invesco Nasdaq 100 ETF',
    'ONEQ': 'Fidelity Nasdaq Composite Index ETF',
    'IWM': 'iShares Russell 2000 ETF',
    'IWB': 'iShares Russell 1000 ETF',
    'IWV': 'iShares Russell 3000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS (SELECT SECTOR SPDRS)
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    'XLRE': 'Real Estate Select Sector SPDR',
    'XLC': 'Communication Services Select Sector SPDR',
    
    # GROWTH ETFs
    'IVW': 'iShares S&P 500 Growth ETF',
    'VONG': 'Vanguard Russell 1000 Growth ETF',
    'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    'VUG': 'Vanguard Growth ETF',
    'SPYG': 'SPDR Portfolio S&P 500 Growth ETF',
    
    # VALUE ETFs
    'IVE': 'iShares S&P 500 Value ETF',
    'VONV': 'Vanguard Russell 1000 Value ETF',
    'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    'VTV': 'Vanguard Value ETF',
    'SPYV': 'SPDR Portfolio S&P 500 Value ETF',
    
    # SIZE ETFs
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    'IJH': 'iShares Core S&P Mid-Cap ETF',
    'IJR': 'iShares Core S&P Small-Cap ETF',
    'MDY': 'SPDR S&P MidCap 400 ETF',
    'SLY': 'SPDR S&P 600 Small Cap ETF',
    'VO': 'Vanguard Mid-Cap ETF',
    'VB': 'Vanguard Small-Cap ETF',
    'SCHA': 'Schwab U.S. Small-Cap ETF',
    'SCHM': 'Schwab U.S. Mid-Cap ETF',
    'VTWO': 'Vanguard Russell 2000 ETF',
    'VTHR': 'Vanguard Russell 3000 ETF',
    'THRK': 'iShares Russell 3000 ETF',
    'SPSM': 'SPDR Portfolio S&P 600 Small Cap ETF',
    'SMLF': 'iShares Small-Cap US Equity Factor ETF',
    
    # NASDAQ SPECIFIC
    'QTEC': 'First Trust Nasdaq-100 Technology Sector Index Fund',
    'QQEW': 'First Trust Nasdaq-100 Equal Weighted Index Fund',
    'QQQG': 'Pacer Nasdaq 100 Top 50 Cash Cows Dividend Growth ETF',
    'QQQV': 'Pacer Nasdaq 100 Top 50 Value ETF',
    
    # DIVIDEND/QUALITY
    'SCHD': 'Schwab U.S. Dividend Equity ETF',
    'VYM': 'Vanguard High Dividend Yield ETF',
    'DVY': 'iShares Select Dividend ETF',
    'QUAL': 'iShares MSCI USA Quality Factor ETF',
    'USMV': 'iShares MSCI USA Min Vol Factor ETF',
    
    # EQUAL WEIGHT
    'EWSC': 'Invesco S&P SmallCap 600 Equal Weight ETF',
    'EWMC': 'Invesco S&P MidCap 400 Equal Weight ETF',
}
universe_symbol_names = {
    'IVV': 'IVV - iShares Core S&P 500 ETF',
    'IJH': 'IJH - iShares Core S&P Mid-Cap ETF',
    'IWM': 'IWM - iShares Russell 2000 ETF',
    'EFA': 'EFA - iShares MSCI EAFE ETF',
    'EEM': 'EEM - iShares MSCI Emerging Markets ETF',
    'AGG': 'AGG - iShares Core U.S. Aggregate Bond ETF',
    'SPTL': 'SPTL - SPDR Portfolio Long Term Treasury ETF',
    'HYG': 'HYG - iShares iBoxx $ High Yield Corporate Bond ETF',
    'SPBO': 'SPBO - SPDR Portfolio Corporate Bond ETF',
    'IYR': 'IYR - iShares U.S. Real Estate ETF',
    'DBC': 'DBC - Invesco DB Commodity Index Tracking Fund',
    'GLD': 'GLD - SPDR Gold Shares',
}

# international_index_data = pd.read_csv('data/processed/index_data.csv', index_col=0, header=[0, 1], parse_dates=True)
commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0, 1], parse_dates=True)
etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
universe_data = pd.read_csv('data/processed/universe_etfs.csv', index_col=0, header=[0, 1], parse_dates=True)

commodity_data_close_cols = commodity_data.columns[commodity_data.columns.get_level_values(1) == 'close']
commodity_close_prices = commodity_data[commodity_data_close_cols].droplevel(1, axis=1).rename(columns=commodity_symbol_names)
commodity_close_prices.columns = [col.replace('/', ' ') for col in commodity_close_prices.columns]

etf_close_cols = etf_data.columns[etf_data.columns.get_level_values(1) == 'close']
etf_close_prices = etf_data[etf_close_cols].droplevel(1, axis=1).rename(columns=etf_symbol_names)

universe_close_cols = universe_data.columns[universe_data.columns.get_level_values(1) == 'close']
universe_close_prices = universe_data[universe_close_cols].droplevel(1, axis=1).rename(columns=universe_symbol_names)

In [4]:
etf_always_disclude = ['Vanguard S&P 500 ETF', 'Real Estate Select Sector SPDR', 'Communication Services Select Sector SPDR']
etf_disclude = [name for name in etf_data.columns if 'Russell 1000' in name or 'Russell 3000' in name]\
                        + ['S&P 500', 'Nasdaq Composite', 'Dow Jones Industrial Average', 'Nasdaq 100', 'Russell 2000']

etf_include = list(set(etf_close_prices.columns.tolist()) - set(etf_always_disclude) - set(etf_disclude))
etf_close_prices = etf_close_prices[etf_include]

us_treasury = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']
int_equity = ['Vanguard Total International Stock ETF', 'Vanguard FTSE Developed Markets ETF',\
                                        'Vanguard FTSE Emerging Markets ETF','Vanguard FTSE Europe ETF',\
                                        'Vanguard FTSE Pacific ETF', 'iShares China Large-Cap ETF',\
                                        'iShares MSCI Japan ETF', 'iShares MSCI India ETF']
us_equity = list(set(etf_include) - set(us_treasury) - set(int_equity))

#### -------------------------------------------------------------------------------------------------

In [5]:
# Create DataFrame of asset names
from KMRF_training_config import *
asset_names_df = pd.DataFrame({
    'universe': get_assets_by_class('universe') + ['']*7,
    'us_equity': get_assets_by_class('us_equity'),
    'commodity': get_assets_by_class('commodity') + ['']*5,
    'int_equity': get_assets_by_class('int_equity') + ['']*11,

})

from pandas import option_context
with option_context('display.max_colwidth', None):
    display(asset_names_df)

,universe,us_equity,commodity,int_equity
0,IVV - iShares Core S&P 500 ETF,SPDR S&P 500 ETF,Gold Futures,Vanguard Total International Stock ETF
1,IJH - iShares Core S&P Mid-Cap ETF,Invesco QQQ Trust,Wheat Futures,Vanguard FTSE Developed Markets ETF
2,IWM - iShares Russell 2000 ETF,iShares Russell 2000 ETF,Corn Futures,Vanguard FTSE Emerging Markets ETF
3,EFA - iShares MSCI EAFE ETF,SPDR Dow Jones Industrial Average ETF,Copper,Vanguard FTSE Europe ETF
4,EEM - iShares MSCI Emerging Markets ETF,Energy Select Sector SPDR,Sugar,Vanguard FTSE Pacific ETF
5,AGG - iShares Core U.S. Aggregate Bond ETF,Financial Select Sector SPDR,Silver Futures,iShares China Large-Cap ETF
6,SPTL - SPDR Portfolio Long Term Treasury ETF,Utilities Select Sector SPDR,US Dollar,iShares MSCI Japan ETF
7,HYG - iShares iBoxx $ High Yield Corporate Bond ETF,Industrial Select Sector SPDR,Soybean Futures,iShares MSCI India ETF
8,SPBO - SPDR Portfolio Corporate Bond ETF,Health Care Select Sector SPDR,Lumber Futures,
9,IYR - iShares U.S. Real Estate ETF,Technology Select Sector SPDR,Live Cattle Futures,


In [6]:
df = etf_close_prices['SPDR S&P 500 ETF'].to_frame().dropna()
rebal_dates = df.loc['2018-12-31':].index[::21]
print(len(rebal_dates))
rebal_dates_to_use = list(reversed(rebal_dates[-48:-4]))
rebal_dates_to_use

82


[Timestamp('2025-06-06 00:00:00'),
 Timestamp('2025-05-07 00:00:00'),
 Timestamp('2025-04-07 00:00:00'),
 Timestamp('2025-03-07 00:00:00'),
 Timestamp('2025-02-05 00:00:00'),
 Timestamp('2025-01-03 00:00:00'),
 Timestamp('2024-12-03 00:00:00'),
 Timestamp('2024-11-01 00:00:00'),
 Timestamp('2024-10-03 00:00:00'),
 Timestamp('2024-09-04 00:00:00'),
 Timestamp('2024-08-05 00:00:00'),
 Timestamp('2024-07-05 00:00:00'),
 Timestamp('2024-06-04 00:00:00'),
 Timestamp('2024-05-03 00:00:00'),
 Timestamp('2024-04-04 00:00:00'),
 Timestamp('2024-03-05 00:00:00'),
 Timestamp('2024-02-02 00:00:00'),
 Timestamp('2024-01-03 00:00:00'),
 Timestamp('2023-12-01 00:00:00'),
 Timestamp('2023-11-01 00:00:00'),
 Timestamp('2023-10-03 00:00:00'),
 Timestamp('2023-09-01 00:00:00'),
 Timestamp('2023-08-03 00:00:00'),
 Timestamp('2023-07-05 00:00:00'),
 Timestamp('2023-06-02 00:00:00'),
 Timestamp('2023-05-03 00:00:00'),
 Timestamp('2023-04-03 00:00:00'),
 Timestamp('2023-03-03 00:00:00'),
 Timestamp('2023-02-

# Fitting KAMA+MSR

In [7]:
def fit_KAMA_MSR(sdte: datetime | str, 
                 edte: datetime | str,
                 asset_names: list[str],
                 n_regimes: int,
                 use_three_state_msr: bool,
                 kama_params: Dict,
                 filter_params: Dict,
                 n_samples: int,
                 burnin: int,
                 thin: int,
                 close_prices: pd.DataFrame,
                 # NEW PARAMETERS FOR IMPROVEMENTS
                 optimize_kama: bool = True,
                 kama_optimization_method: str = 'random',
                 n_random_trials: int = 50,
                 min_regime_duration: int | None = None,
                 duration_enforcement_method: str = 'extend',
                 random_seed: int | None = None,
                 msr_verbose: bool = True) -> Dict[str, 'KAMA_MSR']:
    """
    Fit KAMA+MSR models with improved optimization and duration enforcement.
    
    Parameters:
    -----------
    sdte : datetime | str
        Start date for data
    edte : datetime | str
        End date for data
    asset_names : list[str]
        List of asset names to process
    n_regimes : int
        Number of MSR regimes (2 or 3)
    use_three_state_msr : bool
        Whether to use 3-state MSR
    kama_params : dict
        Initial KAMA parameters (will be overridden if optimize_kama=True)
    filter_params : dict
        Filter parameters
    n_samples : int
        Number of MCMC samples
    burnin : int
        MCMC burnin period
    thin : int
        MCMC thinning
    close_prices : pd.DataFrame
        DataFrame of close prices
    
    NEW PARAMETERS:
    ---------------
    optimize_kama : bool, default=False
        Whether to optimize KAMA parameters using misclassification score
    kama_optimization_method : str, default='random'
        Optimization method: 'random' (fast) or 'coarse_to_fine' (thorough)
    n_random_trials : int, default=50
        Number of random trials for optimization (if method='random')
    optimize_filter : bool, default=False
        Whether to optimize filter parameters
    min_regime_duration : int | None, default=None
        Minimum number of periods a regime must persist
        If None, no duration enforcement
    duration_enforcement_method : str, default='extend'
        Method for duration enforcement: 'extend', 'merge', or 'majority'
    random_seed : int | None, default=None
        Random seed for reproducibility
    msr_verbose : bool, default=True
        Whether to print MSR fitting progress
    
    Returns:
    --------
    Dict[str, KAMA_MSR] : Dictionary of fitted models by asset name
    """
    models = {}
    
    for asset_name in asset_names:
        if asset_name not in close_prices.columns:
            print(f"Asset name {asset_name} not found in provided close prices data. Skipping.")
            continue
        
        print(f"\n{'='*160}")
        print(f"PROCESSING: {asset_name}")
        print(f"{'='*160}")

        # Prepare data
        prices = close_prices[asset_name].dropna()
        prices = prices.loc[sdte:edte]
        
        # Initialize model
        model = KAMA_MSR(
            kama_params=kama_params,
            msr_params={'n_regimes': n_regimes},
            filter_params=filter_params,
            use_three_state_msr=use_three_state_msr,
            random_seed=random_seed  # NEW: Set random seed
        )
        
        # Fit the model with improvements
        model.fit(
            asset_name=asset_name,
            prices=prices,
            
            # NEW: KAMA optimization using misclassification score
            optimize_kama=optimize_kama,
            kama_optimization_method=kama_optimization_method,
            n_random_trials=n_random_trials,
            
            # # Filter optimization (optional)
            # optimize_filter=optimize_filter,
            
            # NEW: Minimum regime duration enforcement
            min_regime_duration=min_regime_duration,
            duration_enforcement_method=duration_enforcement_method,
            
            # MSR parameters
            msr_verbose=msr_verbose,
            n_samples=n_samples,
            burnin=burnin,
            thin=thin
        )
        
        # Print summary statistics
        print(f"\n{'='*80}")
        print(f"SUMMARY FOR {asset_name}")
        print(f"{'='*80}")
        
        if optimize_kama:
            print(f"Optimized KAMA: n={model.kama.n}, n_fast={model.kama.n_fast}, "
                  f"n_slow={model.kama.n_slow}, gamma={model.gamma:.3f}")
        
        if min_regime_duration is not None:
            print(f"\nDuration Statistics:")
            print(model.analyze_regime_durations().to_string(index=False))
        
        print(f"\nRegime Distribution:")
        regime_counts = model.regime_labels.value_counts().sort_index()
        total = len(model.regime_labels)
        for regime, count in regime_counts.items():
            print(f"  Regime {regime}: {count:5d} periods ({100*count/total:5.1f}%)")
        
        n_changes = (model.regime_labels.diff() != 0).sum()
        avg_duration = total / (n_changes + 1)
        print(f"\nRegime Changes: {n_changes}")
        print(f"Average Duration: {avg_duration:.1f} periods")
        
        models[asset_name] = model
        
    print(f"\n{'='*80}")
    print(f"COMPLETED FITTING {len(models)} ASSETS")
    print(f"{'='*80}\n")
    
    return models

def save_KAMA_MSR_models(models: Dict[str, 'KAMA_MSR'], 
                         asset_type_sub_folder: str, 
                         edte_sub_folder: str,
                         save_metadata: bool = True) -> None:
    """
    Save fitted KAMA+MSR models with metadata.
    
    Parameters:
    -----------
    models : Dict[str, KAMA_MSR]
        Dictionary of fitted models
    asset_type_sub_folder : str
        Sub-folder for asset type (e.g., 'us_equity')
    edte_sub_folder : str
        Sub-folder for end date (e.g., '20230101')
    save_metadata : bool, default=True
        Whether to save metadata file with model info
    """
    save_dir = f'saved_models/KAMA_MSR/{asset_type_sub_folder}/{edte_sub_folder}'
    os.makedirs(save_dir, exist_ok=True)
    
    metadata = {}
    
    for asset_name, model in models.items():
        # Clean asset name for filename
        clean_name = asset_name.replace('/', '_').replace('\\', '_')
        n_regimes = model.msr.n_regimes
        total_regimes = n_regimes * 2
        
        # Save model
        filename = f'{clean_name}_KAMA-MSR_{total_regimes}-regimes.pkl'
        filepath = os.path.join(save_dir, filename)
        
        with open(filepath, 'wb') as f:
            pickle.dump(model, f)
        
        print(f"✓ Saved: {filepath}")
        
        # Collect metadata
        if save_metadata:
            metadata[asset_name] = {
                'filename': filename,
                'n_msr_regimes': n_regimes,
                'n_combined_regimes': total_regimes,
                'kama_params': {
                    'n': model.kama.n,
                    'n_fast': model.kama.n_fast,
                    'n_slow': model.kama.n_slow
                },
                'filter_params': {
                    'n_lookback': model.n_lookback,
                    'gamma': model.gamma
                },
                'min_regime_duration': model.min_regime_duration,
                'duration_method': model.duration_method,
                'n_data_points': len(model.regime_labels),
                'regime_distribution': model.regime_labels.value_counts().to_dict(),
                'n_regime_changes': (model.regime_labels.diff() != 0).sum()
            }
    
    # Save metadata
    if save_metadata and metadata:
        metadata_file = os.path.join(save_dir, 'metadata.pkl')
        with open(metadata_file, 'wb') as f:
            pickle.dump(metadata, f)
        print(f"\n✓ Saved metadata: {metadata_file}")

from joblib import Parallel, delayed
import multiprocessing
print("Total CPUs:", multiprocessing.cpu_count())
# use_cpus = int(multiprocessing.cpu_count() / 2)
use_cpus = 12
print("Using CPUs:", use_cpus)

Total CPUs: 20
Using CPUs: 12


In [8]:
def fit_one_asset(asset_name, rebal_date):
    # Inputs
    sdte = datetime(1995, 1, 1)
    edte = rebal_date
    asset_names = [asset_name] # us_equity, us_treasury, int_equity, commodity_close_prices.columns.tolist(), international_index_close_prices.columns.tolist()
    close_prices = universe_close_prices # etf_close_prices, commodity_close_prices, international_index_close_prices
    asset_type_sub_folder = 'universe' # 'us_equity', 'us_treasury', 'int_equity', 'commodity'
    n_regimes=2
    use_three_state_msr = (n_regimes == 3)
    kama_params = {'n': 20, 'n_fast': 5, 'n_slow': 30}
    filter_params = {'n_lookback': False, 'gamma': 1}
    n_samples=1000
    burnin=200
    thin=1
    min_regime_duration = 1

    models = fit_KAMA_MSR(
        sdte=sdte,
        edte=edte,
        asset_names=asset_names,
        n_regimes=n_regimes,
        use_three_state_msr=False,
        kama_params=kama_params,
        filter_params=filter_params,
        n_samples=n_samples,
        burnin=burnin,
        thin=thin,
        close_prices=close_prices,
        # Full optimization
        optimize_kama=True,
        kama_optimization_method='coarse_to_fine',              
        min_regime_duration=min_regime_duration,
        duration_enforcement_method='merge',
        random_seed=1010,
        msr_verbose=True
    )

    save_KAMA_MSR_models(models, asset_type_sub_folder, edte.strftime('%Y%m%d'), save_metadata=False)

for i, rebal_date in enumerate(rebal_dates_to_use):
    print(f'Fitting for rebalance date {i+1}/{len(rebal_dates_to_use)}:', rebal_date)
    results = Parallel(n_jobs=use_cpus)(
        delayed(fit_one_asset)(asset, rebal_date) for asset in asset_names_df['universe'].tolist()
    )
    !git add .
    !git commit -m "Universe KAMA+MSR fitting with end date ({rebal_date})"
    !git push

Fitting for rebalance date 1/44: 2025-06-06 00:00:00


[main 698a80a] Universe KAMA+MSR fitting with end date (2025-06-06 00:00:00)
 13 files changed, 11 insertions(+), 14 deletions(-)
 create mode 100644 saved_models/KAMA_MSR/universe/20250606/AGG - iShares Core U.S. Aggregate Bond ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250606/DBC - Invesco DB Commodity Index Tracking Fund_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250606/EEM - iShares MSCI Emerging Markets ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250606/EFA - iShares MSCI EAFE ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250606/GLD - SPDR Gold Shares_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250606/HYG - iShares iBoxx $ High Yield Corporate Bond ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250606/IJH - iShares Core S&P Mid-Cap ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 s

To https://github.com/jessego20/FE800_project_code.git
   3567ad2..698a80a  main -> main


Fitting for rebalance date 2/44: 2025-05-07 00:00:00
[main 525d5f9] Universe KAMA+MSR fitting with end date (2025-05-07 00:00:00)
 12 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 saved_models/KAMA_MSR/universe/20250507/AGG - iShares Core U.S. Aggregate Bond ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250507/DBC - Invesco DB Commodity Index Tracking Fund_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250507/EEM - iShares MSCI Emerging Markets ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250507/EFA - iShares MSCI EAFE ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250507/GLD - SPDR Gold Shares_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250507/HYG - iShares iBoxx $ High Yield Corporate Bond ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20250507/IJH - iShares Core S&P Mid-C

To https://github.com/jessego20/FE800_project_code.git
   698a80a..525d5f9  main -> main


...

To https://github.com/jessego20/FE800_project_code.git
   99f9ece..5e32ee5  main -> main


Fitting for rebalance date 43/44: 2021-11-30 00:00:00
[main 22af057] Universe KAMA+MSR fitting with end date (2021-11-30 00:00:00)
 12 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 saved_models/KAMA_MSR/universe/20211130/AGG - iShares Core U.S. Aggregate Bond ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211130/DBC - Invesco DB Commodity Index Tracking Fund_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211130/EEM - iShares MSCI Emerging Markets ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211130/EFA - iShares MSCI EAFE ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211130/GLD - SPDR Gold Shares_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211130/HYG - iShares iBoxx $ High Yield Corporate Bond ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211130/IJH - iShares Core S&P Mid-

To https://github.com/jessego20/FE800_project_code.git
   5e32ee5..22af057  main -> main


Fitting for rebalance date 44/44: 2021-10-29 00:00:00
[main df79d1e] Universe KAMA+MSR fitting with end date (2021-10-29 00:00:00)
 12 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 saved_models/KAMA_MSR/universe/20211029/AGG - iShares Core U.S. Aggregate Bond ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211029/DBC - Invesco DB Commodity Index Tracking Fund_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211029/EEM - iShares MSCI Emerging Markets ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211029/EFA - iShares MSCI EAFE ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211029/GLD - SPDR Gold Shares_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211029/HYG - iShares iBoxx $ High Yield Corporate Bond ETF_KAMA-MSR_4-regimes.pkl
 create mode 100644 saved_models/KAMA_MSR/universe/20211029/IJH - iShares Core S&P Mid-

To https://github.com/jessego20/FE800_project_code.git
   22af057..df79d1e  main -> main


In [46]:
# for rebal_date in rebal_dates[3:]:
#     # Inputs
#     sdte = datetime(1995, 1, 1)
#     edte = rebal_date
#     asset_names = asset_names_df['us_equity'].tolist() # us_equity, us_treasury, int_equity, commodity_close_prices.columns.tolist(), international_index_close_prices.columns.tolist()
#     close_prices = etf_close_prices # etf_close_prices, commodity_close_prices, international_index_close_prices
#     asset_type_sub_folder = 'us_equity' # 'us_equity', 'us_treasury', 'int_equity', 'commodity'
#     n_regimes=2
#     use_three_state_msr = (n_regimes == 3)
#     kama_params = {'n': 20, 'n_fast': 5, 'n_slow': 30}
#     filter_params = {'n_lookback': False, 'gamma': 1}
#     n_samples=1000
#     burnin=200
#     thin=1
#     min_regime_duration = 1

#     models = fit_KAMA_MSR(
#         sdte=sdte,
#         edte=edte,
#         asset_names=asset_names,
#         n_regimes=n_regimes,
#         use_three_state_msr=False,
#         kama_params=kama_params,
#         filter_params=filter_params,
#         n_samples=n_samples,
#         burnin=burnin,
#         thin=thin,
#         close_prices=close_prices,
#         # Full optimization
#         optimize_kama=True,
#         kama_optimization_method='coarse_to_fine',              
#         min_regime_duration=min_regime_duration,
#         duration_enforcement_method='merge',
#         random_seed=1010,
#         msr_verbose=True
#     )

#     save_KAMA_MSR_models(models, asset_type_sub_folder, edte.strftime('%Y%m%d'), save_metadata=False)

#     # !git add .
#     # !git commit -m f"Updated KAMA+MSR fitting with new end date ({edte})"
#     # !git push

# Batch Model Fit

In [12]:
# # Inputs
# sdte = datetime(1995, 1, 1)
# edte = datetime(2019, 1, 1)
# n_regimes=2
# use_three_state_msr = (n_regimes == 3)
# kama_params = {'n': 20, 'n_fast': 5, 'n_slow': 30}
# filter_params = {'n_lookback': False, 'gamma': 0.75}
# n_samples=1000
# burnin=200
# thin=1
# min_regime_duration=1

# def fit_KAMA_MSR(sdte: datetime | str, 
#                  edte: datetime | str,
#                  asset_names: list[str],
#                  n_regimes: int,
#                  use_three_state_msr: bool,
#                  kama_params: Dict,
#                  filter_params: Dict,
#                  n_samples: int,
#                  burnin: int,
#                  thin: int,
#                  close_prices: pd.DataFrame,
#                  # NEW PARAMETERS FOR IMPROVEMENTS
#                  optimize_kama: bool = True,
#                  kama_optimization_method: str = 'random',
#                  n_random_trials: int = 50,
#                  min_regime_duration: int | None = None,
#                  duration_enforcement_method: str = 'extend',
#                  random_seed: int | None = None,
#                  msr_verbose: bool = True) -> Dict[str, 'KAMA_MSR']:
#     """
#     Fit KAMA+MSR models with improved optimization and duration enforcement.
    
#     Parameters:
#     -----------
#     sdte : datetime | str
#         Start date for data
#     edte : datetime | str
#         End date for data
#     asset_names : list[str]
#         List of asset names to process
#     n_regimes : int
#         Number of MSR regimes (2 or 3)
#     use_three_state_msr : bool
#         Whether to use 3-state MSR
#     kama_params : dict
#         Initial KAMA parameters (will be overridden if optimize_kama=True)
#     filter_params : dict
#         Filter parameters
#     n_samples : int
#         Number of MCMC samples
#     burnin : int
#         MCMC burnin period
#     thin : int
#         MCMC thinning
#     close_prices : pd.DataFrame
#         DataFrame of close prices
    
#     NEW PARAMETERS:
#     ---------------
#     optimize_kama : bool, default=False
#         Whether to optimize KAMA parameters using misclassification score
#     kama_optimization_method : str, default='random'
#         Optimization method: 'random' (fast) or 'coarse_to_fine' (thorough)
#     n_random_trials : int, default=50
#         Number of random trials for optimization (if method='random')
#     optimize_filter : bool, default=False
#         Whether to optimize filter parameters
#     min_regime_duration : int | None, default=None
#         Minimum number of periods a regime must persist
#         If None, no duration enforcement
#     duration_enforcement_method : str, default='extend'
#         Method for duration enforcement: 'extend', 'merge', or 'majority'
#     random_seed : int | None, default=None
#         Random seed for reproducibility
#     msr_verbose : bool, default=True
#         Whether to print MSR fitting progress
    
#     Returns:
#     --------
#     Dict[str, KAMA_MSR] : Dictionary of fitted models by asset name
#     """
#     models = {}
    
#     for asset_name in asset_names:
#         if asset_name not in close_prices.columns:
#             print(f"Asset name {asset_name} not found in provided close prices data. Skipping.")
#             continue
        
#         print(f"\n{'='*160}")
#         print(f"PROCESSING: {asset_name}")
#         print(f"{'='*160}")

#         # Prepare data
#         prices = close_prices[asset_name].dropna()
#         prices = prices.loc[sdte:edte]
        
#         # Initialize model
#         model = KAMA_MSR(
#             kama_params=kama_params,
#             msr_params={'n_regimes': n_regimes},
#             filter_params=filter_params,
#             use_three_state_msr=use_three_state_msr,
#             random_seed=random_seed  # NEW: Set random seed
#         )
        
#         # Fit the model with improvements
#         model.fit(
#             asset_name=asset_name,
#             prices=prices,
            
#             # NEW: KAMA optimization using misclassification score
#             optimize_kama=optimize_kama,
#             kama_optimization_method=kama_optimization_method,
#             n_random_trials=n_random_trials,
            
#             # # Filter optimization (optional)
#             # optimize_filter=optimize_filter,
            
#             # NEW: Minimum regime duration enforcement
#             min_regime_duration=min_regime_duration,
#             duration_enforcement_method=duration_enforcement_method,
            
#             # MSR parameters
#             msr_verbose=msr_verbose,
#             n_samples=n_samples,
#             burnin=burnin,
#             thin=thin
#         )
        
#         # Print summary statistics
#         print(f"\n{'='*80}")
#         print(f"SUMMARY FOR {asset_name}")
#         print(f"{'='*80}")
        
#         if optimize_kama:
#             print(f"Optimized KAMA: n={model.kama.n}, n_fast={model.kama.n_fast}, "
#                   f"n_slow={model.kama.n_slow}, gamma={model.gamma:.3f}")
        
#         if min_regime_duration is not None:
#             print(f"\nDuration Statistics:")
#             print(model.analyze_regime_durations().to_string(index=False))
        
#         print(f"\nRegime Distribution:")
#         regime_counts = model.regime_labels.value_counts().sort_index()
#         total = len(model.regime_labels)
#         for regime, count in regime_counts.items():
#             print(f"  Regime {regime}: {count:5d} periods ({100*count/total:5.1f}%)")
        
#         n_changes = (model.regime_labels.diff() != 0).sum()
#         avg_duration = total / (n_changes + 1)
#         print(f"\nRegime Changes: {n_changes}")
#         print(f"Average Duration: {avg_duration:.1f} periods")
        
#         models[asset_name] = model
        
#     print(f"\n{'='*80}")
#     print(f"COMPLETED FITTING {len(models)} ASSETS")
#     print(f"{'='*80}\n")
    
#     return models

# def save_KAMA_MSR_models(models: Dict[str, 'KAMA_MSR'], 
#                          asset_type_sub_folder: str, 
#                          edte_sub_folder: str,
#                          save_metadata: bool = True) -> None:
#     """
#     Save fitted KAMA+MSR models with metadata.
    
#     Parameters:
#     -----------
#     models : Dict[str, KAMA_MSR]
#         Dictionary of fitted models
#     asset_type_sub_folder : str
#         Sub-folder for asset type (e.g., 'international_index')
#     edte_sub_folder : str
#         Sub-folder for end date (e.g., '20230101')
#     save_metadata : bool, default=True
#         Whether to save metadata file with model info
#     """
#     save_dir = f'saved_models/KAMA_MSR/{asset_type_sub_folder}/{edte_sub_folder}'
#     os.makedirs(save_dir, exist_ok=True)
    
#     metadata = {}
    
#     for asset_name, model in models.items():
#         # Clean asset name for filename
#         clean_name = asset_name.replace('/', '_').replace('\\', '_')
#         n_regimes = model.msr.n_regimes
#         total_regimes = n_regimes * 2
        
#         # Save model
#         filename = f'{clean_name}_KAMA-MSR_{total_regimes}-regimes.pkl'
#         filepath = os.path.join(save_dir, filename)
        
#         with open(filepath, 'wb') as f:
#             pickle.dump(model, f)
        
#         print(f"✓ Saved: {filepath}")
        
#         # Collect metadata
#         if save_metadata:
#             metadata[asset_name] = {
#                 'filename': filename,
#                 'n_msr_regimes': n_regimes,
#                 'n_combined_regimes': total_regimes,
#                 'kama_params': {
#                     'n': model.kama.n,
#                     'n_fast': model.kama.n_fast,
#                     'n_slow': model.kama.n_slow
#                 },
#                 'filter_params': {
#                     'n_lookback': model.n_lookback,
#                     'gamma': model.gamma
#                 },
#                 'min_regime_duration': model.min_regime_duration,
#                 'duration_method': model.duration_method,
#                 'n_data_points': len(model.regime_labels),
#                 'regime_distribution': model.regime_labels.value_counts().to_dict(),
#                 'n_regime_changes': (model.regime_labels.diff() != 0).sum()
#             }
    
#     # Save metadata
#     if save_metadata and metadata:
#         metadata_file = os.path.join(save_dir, 'metadata.pkl')
#         with open(metadata_file, 'wb') as f:
#             pickle.dump(metadata, f)
#         print(f"\n✓ Saved metadata: {metadata_file}")

# for asset_names, close_prices, asset_type_sub_folder in\
#     zip([commodity_close_prices.columns.tolist(), us_equity, us_treasury, int_equity],
#         [commodity_close_prices, us_traded_close_prices, us_traded_close_prices, us_traded_close_prices],
#         ['commodity', 'us_equity', 'us_treasury', 'int_equity']):
    
#     models = fit_KAMA_MSR(
#         sdte=sdte,
#         edte=edte,
#         asset_names=asset_names,
#         n_regimes=n_regimes,
#         use_three_state_msr=False,
#         kama_params=kama_params,
#         filter_params=filter_params,
#         n_samples=n_samples,
#         burnin=burnin,
#         thin=thin,
#         close_prices=close_prices,
#         # Full optimization
#         optimize_kama=True,
#         kama_optimization_method='coarse_to_fine',              
#         min_regime_duration=min_regime_duration,
#         duration_enforcement_method='merge',
#         random_seed=1010,
#         msr_verbose=True
#     )

#     save_KAMA_MSR_models(models, asset_type_sub_folder, edte.strftime('%Y%m%d'), save_metadata=False)

# #     !git add .
# #     !git commit -m f"Auto-commit after batch model fit for {asset_type_sub_folder} ending {edte.strftime('%Y%m%d')}"
# #     !git push